# Deep Learning Training on SageMaker
This notebook demonstrates how to run a deep learning training job on AWS SageMaker using your custom training script in `src/train.py`.

In [ ]:
# Install required dependencies (if running locally, uncomment below)
# !pip install accelerate ipykernel ipywidgets jupyter loguru pandas ruff scikit-learn torch transformers sagemaker

## Set up SageMaker environment and permissions

In [1]:
import sagemaker
from sagemaker import get_execution_role
import boto3
import os

role = get_execution_role()  # If running in SageMaker notebook instance
session = sagemaker.Session()
bucket = "hela-training-test"  # Replace with your S3 bucket name
s3_data_path = f"s3://{bucket}/data"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Upload your training script and requirements to S3 (if needed)

In [ ]:
# If your training script is not already in S3, upload it
# session.upload_data(path='src', bucket=bucket, key_prefix='src')

## Define the SageMaker PyTorch Estimator

In [2]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="src/train.py",
    source_dir=".",
    role=role,
    framework_version="2.0",  # Match your torch version if needed
    py_version="py310",
    instance_count=1,
    instance_type="ml.g4dn.xlarge",  # Change as needed
    hyperparameters={
        "data_path": s3_data_path,
        # Add other hyperparameters if needed
    },
    output_path=f"s3://{bucket}/output",
)

## Launch the training job

In [3]:
estimator.fit({"train": s3_data_path})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2025-08-25-04-10-21-714


2025-08-25 04:10:22 Starting - Starting the training job
2025-08-25 04:10:22 Pending - Training job waiting for capacity...
2025-08-25 04:10:55 Pending - Preparing the instances for training...
2025-08-25 04:11:23 Downloading - Downloading input data...
2025-08-25 04:11:48 Downloading - Downloading the training image.....................
2025-08-25 04:15:06 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2025-08-25 04:15:20,066 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-08-25 04:15:20,086 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-08-25 04:15:20,096 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-08-25 04:15:20,104 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-08-25 04:15:22

## Monitor and retrieve training results

In [ ]:
# After training, you can access model artifacts in S3
print("Model artifacts saved to:", estimator.model_data)